# Exploratory Data Analysis - Heart Disease

This notebook explores the dataset before applying IQR and Z-Score outlier detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config.config import Data_Path

In [ ]:
df = pd.read_csv(Data_Path)
df.head()

## 1. Basic Information

In [ ]:
print('Shape:', df.shape)
print('\nData types and missing values:')
df.info()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_percent = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    'Missing': missing,
    'Percentage': missing_percent
})
missing_table

## 2. Numerical Features

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns
numeric_columns

In [ ]:
df[numeric_columns].describe().T

In [ ]:
df[numeric_columns].nunique().sort_values()

## 3. Target Distribution

In [ ]:
df['Heart Disease Status'].value_counts()

## 4. Distribution and Skewness

In [ ]:
df[numeric_columns].hist(figsize=(12, 10), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
df[numeric_columns].skew().sort_values()

## 5. IQR Outlier Detection

We use the standard 1.5 × IQR rule. Extreme fences are not used.

In [ ]:
iqr_results = []

for column in numeric_columns:
    data = df[column].dropna()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((data < lower) | (data > upper)).sum()

    iqr_results.append([column, q1, q3, iqr, lower, upper, count])

iqr_table = pd.DataFrame(
    iqr_results,
    columns=['Feature', 'Q1', 'Q3', 'IQR', 'Lower Fence', 'Upper Fence', 'Outlier Count']
)

iqr_table

## 6. Z-Score Outlier Detection

A value is treated as a potential outlier when |Z| > 3.

In [ ]:
zscore_results = []

for column in numeric_columns:
    data = df[column].dropna()
    mean = data.mean()
    std = data.std()
    z = (data - mean) / std
    count = (z.abs() > 3).sum()
    zscore_results.append([column, mean, std, count])

zscore_table = pd.DataFrame(
    zscore_results,
    columns=['Feature', 'Mean', 'Std', 'Z-Score Outlier Count']
)

zscore_table

## 7. Comparison

In [ ]:
comparison = iqr_table[['Feature', 'Outlier Count']].copy()
comparison = comparison.rename(columns={'Outlier Count': 'IQR Outliers'})
comparison['Z-Score Outliers'] = zscore_table['Z-Score Outlier Count'].values
comparison['Total Outliers'] = comparison['IQR Outliers'] + comparison['Z-Score Outliers']
comparison

## Conclusion

The exploratory analysis shows that the numerical features do not contain potential outliers under the selected IQR and Z-Score rules. Therefore, no trimming or capping is applied to these numerical features.